# TRAMA Visual Factory — Free GPU Probe v0.2

Hardware classification only. No image models are downloaded.

The probe separates a 16-GB-class **Q4 candidate** from the heavier FP8 reference profile.

In [ ]:
import json, os, shutil, subprocess

def sh(cmd):
    return subprocess.check_output(cmd, shell=True, text=True).strip()

gpu_present = shutil.which('nvidia-smi') is not None
gpu_name = None
vram_gib = 0.0

if gpu_present:
    gpu_name = sh("nvidia-smi --query-gpu=name --format=csv,noheader | head -1")
    vram_mib = float(sh("nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits | head -1"))
    vram_gib = vram_mib / 1024

mem_kib = int(sh("awk '/MemTotal/ {print $2}' /proc/meminfo"))
ram_gib = mem_kib / 1024 / 1024
disk = shutil.disk_usage('/content' if os.path.exists('/content') else '/')
disk_free_gib = disk.free / (1024**3)

if not gpu_present:
    decision = 'NO_GPU'
elif vram_gib >= 22 and ram_gib >= 20 and disk_free_gib >= 35:
    decision = 'REFERENCE_FP8_CANDIDATE'
elif vram_gib >= 14 and ram_gib >= 11 and disk_free_gib >= 35:
    decision = 'T4_Q4_CANDIDATE'
else:
    decision = 'INSUFFICIENT'

receipt = {
    'schemaVersion': 'trama.visual-factory.free-gpu-probe/v0.2',
    'candidateId': 'MZ-VF-001',
    'adapter': 'google-colab-free',
    'gpuPresent': gpu_present,
    'gpuName': gpu_name,
    'vramGiB': round(vram_gib, 2),
    'ramGiB': round(ram_gib, 2),
    'diskFreeGiB': round(disk_free_gib, 2),
    'decision': decision,
    'paidComputeAuthorized': False,
    'nextStep': {
        'NO_GPU': 'Enable a free GPU runtime and rerun the probe.',
        'T4_Q4_CANDIDATE': 'Proceed only to the Q4 quantized smoke test; not yet production-qualified.',
        'REFERENCE_FP8_CANDIDATE': 'Proceed to the FP8/reference smoke test.',
        'INSUFFICIENT': 'Stop. Do not download the model package on this runtime.'
    }[decision]
}

print(json.dumps(receipt, indent=2))
with open('trama-visual-factory-gpu-probe.json', 'w', encoding='utf-8') as f:
    json.dump(receipt, f, indent=2)
